# CDP 2025 — St. Petersburg Response Extractor

This notebook reads a CDP Excel file and pulls out **all questions + St. Petersburg's responses** (disclosure no. `49172`) into a clean Excel sheet.

> **Note:** St. Petersburg is a *city* — its data lives in the **CDP Cities dataset**, not the States & Regions file.  
> Update `INPUT_FILE` in the Config cell once you have the correct file.

## Step 1 — Install Dependencies

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "pandas", "-q"])
print("Ready.")

Ready.


## Step 2 — Configuration

Set the input file(s) per year and St. Petersburg's disclosure number here. Add/remove entries in `INPUT_FILES` to control which years are included.

In [2]:
INPUT_FILES    = {
    2025: "cdp_cities_data/2025_Full_Cities_Public_Data_Separated_by_Question.xlsx",
    2024: "cdp_cities_data/2024_Full_Cities_Public_Data_Separated_By_Question_CDP_ICLEI_Track.xlsx",
    2023: "cdp_cities_data/2023_Full_GCoM_Cities_Data_Separated_by_Question_151223.xlsx",
    2022: "cdp_cities_data/2022_Full_Cities_Data_Separated_by_Question.xlsx",
    2021: "cdp_cities_data/2021_Full_Public_Cities_Data_Separated_by_Question.xlsx",
}
ST_PETERSBURG_DISC_NO = 49172   # St. Petersburg's org number — same value across all years, but under a
                          # different column name depending on schema (see Step 3)
OUTPUT_FILE    = "St_Petersburg_Responses.xlsx"
SKIP_SHEETS    = {"Introduction", "Summary", "Summary Data"}  # non-question sheets to ignore

## Step 3 — Read & Filter

We use `pandas` to read all sheets at once, then keep only rows where the disclosing org is St. Petersburg.

The 2025/2024 files and the 2022/2023 files use **different layouts**, so we branch on which schema each sheet matches:

- **Schema A (2025, 2024):** a `cdp_disclosing_org_number` column identifies the org; metadata is in fixed named columns (`question_number`, `question_text`, `row_name`, …); response columns are prefixed `col1_`, `col2_`, …
- **Schema B (2023, 2022, 2021):** an `Account Number` column identifies the org; metadata columns are `Section`, `RowNumber`, `RowName`, …; the question number, question text, and sub-field name are all packed into each response column's header, e.g. `"1.2 C1 - Provide details on the most significant climate hazards... - Climate-related hazards^"` (split on `" - "`).

Both branches emit the same flattened record shape (`Question`, `Year`, `Question Text`, `Sub-fields`, `Mumbai Response`), one row per sub-field.

In [3]:
import pandas as pd
import re

# --- Schema A (2025 / 2024): metadata columns present in every question sheet ---
META_COLS_A = {
    "disclosure_cycle", "cdp_requesting_org_number", "requesting_organization",
    "cdp_disclosing_org_number", "disclosing_organization", "disclosing_org_type",
    "cdp_region", "discloser_country_or_area", "public_status",
    "question_number", "question_text", "row_order", "row_name"
}

# --- Schema B (2023 / 2022 / 2021): metadata columns present in every question sheet ---
META_COLS_B = {
    "Questionnaire Name", "Account Number", "Account Name", "Country/Areas",
    "CDP Region", "Authorities", "ParentSection", "Section", "RowNumber", "RowName"
}

records = []

for year, input_file in INPUT_FILES.items():
    print(f"Reading {year}: {input_file} ...")
    all_sheets = pd.read_excel(input_file, sheet_name=None)

    year_count = 0
    for sheet_name, df in all_sheets.items():
        if sheet_name in SKIP_SHEETS:
            continue

        if "cdp_disclosing_org_number" in df.columns:
            # ---------- Schema A ----------
            st_petersburg = df[df["cdp_disclosing_org_number"] == ST_PETERSBURG_DISC_NO]
            if st_petersburg.empty:
                continue
            response_cols = [c for c in df.columns if c not in META_COLS_A]

            for _, row in st_petersburg.iterrows():
                q_num  = row.get("question_number", sheet_name)
                q_text = row.get("question_text", "")
                r_name = row.get("row_name", "")

                label = str(q_num) if pd.notna(q_num) else sheet_name
                if pd.notna(r_name) and str(r_name).strip():
                    label += f" – {r_name}"
                label_text = str(q_text) if pd.notna(q_text) else ""

                # One row per sub-field — Question / Question Text repeat for each
                for col in response_cols:
                    val = row.get(col)
                    if pd.notna(val) and str(val).strip():
                        # strip "col1_", "col2_" … prefix using "_" as the cut point
                        clean_col = re.sub(r"^col\d+_", "", col)

                        records.append({
                            "Question":        label,
                            "Year":            year,
                            "Question Text":   label_text,
                            "Sub-fields":      clean_col,
                            "St. Petersburg Response": str(val)
                        })
                        year_count += 1

        elif "Account Number" in df.columns:
            # ---------- Schema B ----------
            st_petersburg = df[df["Account Number"] == ST_PETERSBURG_DISC_NO]
            if st_petersburg.empty:
                continue
            response_cols = [c for c in df.columns if c not in META_COLS_B]
            multi_row = len(st_petersburg) > 1  # only disambiguate rows when this sheet has several
            label_base = f"Q{sheet_name}" if str(sheet_name)[0].isdigit() else str(sheet_name)

            for _, row in st_petersburg.iterrows():
                r_name = row.get("RowName", "")
                r_num  = row.get("RowNumber", "")

                label = label_base
                if pd.notna(r_name) and str(r_name).strip():
                    label += f" – {str(r_name).strip()}"
                elif multi_row and pd.notna(r_num):
                    label += f" – Row {int(r_num)}"

                # Column headers look like "1.2 C1 - <question text> - <sub-field>^"
                # (or "1.1 - <question text>" when there's only one field)
                for col in response_cols:
                    val = row.get(col)
                    if pd.notna(val) and str(val).strip():
                        parts = [p.strip().rstrip("^").strip() for p in col.split(" - ")]
                        q_text   = parts[1] if len(parts) >= 2 else parts[0]
                        subfield = parts[2] if len(parts) == 3 else q_text

                        records.append({
                            "Question":        label,
                            "Year":            year,
                            "Question Text":   q_text,
                            "Sub-fields":      subfield,
                            "St. Petersburg Response": str(val)
                        })
                        year_count += 1

    print(f"  -> {year_count} rows for St. Petersburg (org/disclosure no. {ST_PETERSBURG_DISC_NO}).")

print(f"\nTotal: {len(records)} rows across {len(INPUT_FILES)} year(s).")
if not records:
    print("\n  St. Petersburg not found — please check INPUT_FILES / ST_PETERSBURG_DISC_NO.")

Reading 2025: cdp_cities_data/2025_Full_Cities_Public_Data_Separated_by_Question.xlsx ...
  -> 537 rows for St. Petersburg (org/disclosure no. 49172).
Reading 2024: cdp_cities_data/2024_Full_Cities_Public_Data_Separated_By_Question_CDP_ICLEI_Track.xlsx ...
  -> 579 rows for St. Petersburg (org/disclosure no. 49172).
Reading 2023: cdp_cities_data/2023_Full_GCoM_Cities_Data_Separated_by_Question_151223.xlsx ...
  -> 707 rows for St. Petersburg (org/disclosure no. 49172).
Reading 2022: cdp_cities_data/2022_Full_Cities_Data_Separated_by_Question.xlsx ...
  -> 655 rows for St. Petersburg (org/disclosure no. 49172).
Reading 2021: cdp_cities_data/2021_Full_Public_Cities_Data_Separated_by_Question.xlsx ...
  -> 799 rows for St. Petersburg (org/disclosure no. 49172).

Total: 3277 rows across 5 year(s).


## Step 4 — Save to Excel

We write the collected records to a styled Excel file with:
- A dark blue header row
- Alternating row shading for readability
- Text wrapping so multi-line responses display correctly

In [4]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

if not records:
    print("Nothing to save — re-run Step 3 after updating INPUT_FILES.")
else:
    df_out = pd.DataFrame(records, columns=["Question", "Year", "Question Text", "Sub-fields", "St. Petersburg Response"])

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "St. Petersburg Responses"

    # --- Header row ---
    ws.append(list(df_out.columns))
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.fill      = PatternFill("solid", fgColor="1F4E79")
        cell.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 28

    # --- Data rows ---
    alt_fill = PatternFill("solid", fgColor="DCE6F1")
    for i, row_data in enumerate(df_out.itertuples(index=False), start=2):
        ws.append(list(row_data))
        for cell in ws[i]:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
            if i % 2 == 0:
                cell.fill = alt_fill

    # --- Column widths ---
    ws.column_dimensions["A"].width = 25   # Question
    ws.column_dimensions["B"].width = 8    # Year
    ws.column_dimensions["C"].width = 55   # Question Text
    ws.column_dimensions["D"].width = 50   # Sub-fields
    ws.column_dimensions["E"].width = 55   # St. Petersburg Response

    wb.save(OUTPUT_FILE)
    print(f"Saved {len(records)} rows → '{OUTPUT_FILE}'")

Saved 3277 rows → 'St_Petersburg_Responses.xlsx'
